In [12]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split
import os
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score
from models import *


In [13]:
# Evaluation function
def evaluate_test(model, test_loader):
    model.eval()
    actuals, predictions = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.view(images.size(0), -1)
            labels = labels.float().view(-1, 1)
            
            outputs = model(images)
            predictions.append(outputs.numpy())
            actuals.append(labels.numpy())
    
    predictions = np.vstack(predictions)
    actuals = np.vstack(actuals)
    
    # Calculate regression metrics
    mse = mean_squared_error(actuals, predictions)
    r2 = r2_score(actuals, predictions)
    
    # Convert regression outputs to classification by rounding to the nearest integer
    predicted_classes = np.round(predictions).astype(int)
    actual_classes = actuals.astype(int)
    
    # Ensure predictions are within the valid range [0, 9]
    predicted_classes = np.clip(predicted_classes, 0, 9)
    
    # Calculate accuracy
    accuracy = accuracy_score(actual_classes, predicted_classes)
    
    print(f'Mean Squared Error: {mse:.4f}')
    print(f'R² Score: {r2:.4f}')
    print(f'Accuracy: {accuracy:.4f}')
    
    return mse, r2, accuracy

# Function to evaluate the model
def evaluate(model, data_loader, criterion):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for images, labels in data_loader:
            images = images.view(images.size(0), -1)
            labels = labels.float().view(-1, 1)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * images.size(0)
    
    avg_loss = total_loss / len(data_loader.dataset)
    return avg_loss

# Training function with validation
def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=10):
    best_model = None
    best_val_loss = float('inf')
    final_model = None
    
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        
        for images, labels in train_loader:
            images = images.view(images.size(0), -1)  # Flatten the image
            labels = labels.float().view(-1, 1)  # Convert labels to float for regression
            
            # Zero the parameter gradients
            optimizer.zero_grad()
            
            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            # Backward pass and optimize
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * images.size(0)
        
        # Calculate average training loss
        epoch_loss = running_loss / len(train_loader.dataset)
        
        # Evaluate on the validation set
        val_loss = evaluate(model, val_loader, criterion)
        
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}, Val Loss: {val_loss:.4f}')
        
        # Check if this is the best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model = model.state_dict()  # Save the best model
            
        # Save the final model at the end of training
        final_model = model.state_dict()
    
    # Load the best model for return
    model.load_state_dict(best_model)
    best_model_instance = model
    
    return best_model_instance, final_model

In [17]:
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score

# Function to flatten images based on their dimensionality
def flatten_images(images):
    if images.dim() == 4:  # Images have shape (batch_size, channels, height, width)
        return images.view(images.size(0), -1)
    else:
        raise ValueError("Input images have an unexpected shape. Expected 4D tensor (batch_size, channels, height, width).")

# Evaluation function
def evaluate_test(model, test_loader, num_classes=10):
    model.eval()
    actuals, predictions = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            images = flatten_images(images)
            labels = labels.float().view(-1, 1)
            
            outputs = model(images)
            predictions.append(outputs.numpy())
            actuals.append(labels.numpy())
    
    predictions = np.vstack(predictions)
    actuals = np.vstack(actuals)
    
    # Calculate regression metrics
    mse = mean_squared_error(actuals, predictions)
    r2 = r2_score(actuals, predictions)
    
    # Convert regression outputs to classification by rounding to the nearest integer
    predicted_classes = np.round(predictions).astype(int)
    actual_classes = actuals.astype(int)
    
    # Ensure predictions are within the valid range [0, num_classes-1]
    predicted_classes = np.clip(predicted_classes, 0, num_classes - 1)
    
    # Calculate accuracy
    accuracy = accuracy_score(actual_classes, predicted_classes)
    
    print(f'Mean Squared Error: {mse:.4f}')
    print(f'R² Score: {r2:.4f}')
    print(f'Accuracy: {accuracy:.4f}')
    
    return mse, r2, accuracy

# Function to evaluate the model
def evaluate(model, data_loader, criterion):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for images, labels in data_loader:
            images = flatten_images(images)
            labels = labels.float().view(-1, 1)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * images.size(0)
    
    avg_loss = total_loss / len(data_loader.dataset)
    return avg_loss

# Training function with validation
def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=10):
    best_model = None
    best_val_loss = float('inf')
    final_model = None
    
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        
        for images, labels in train_loader:
            images = flatten_images(images)  # Flatten the image according to its dimensionality
            labels = labels.float().view(-1, 1)  # Convert labels to float for regression
            
            # Zero the parameter gradients
            optimizer.zero_grad()
            
            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            # Backward pass and optimize
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * images.size(0)
        
        # Calculate average training loss
        epoch_loss = running_loss / len(train_loader.dataset)
        
        # Evaluate on the validation set
        val_loss = evaluate(model, val_loader, criterion)
        
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}, Val Loss: {val_loss:.4f}')
        
        # Check if this is the best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model = model.state_dict()  # Save the best model
            
        # Save the final model at the end of training
        final_model = model.state_dict()
    
    # Load the best model for return
    model.load_state_dict(best_model)
    best_model_instance = model
    
    return best_model_instance, final_model

In [20]:

    # Load dataset
    train_dataset = torch.load('datasets/cifar10_train.pt')
    test_dataset = torch.load('datasets/cifar10_test.pt')

    # Check if using MNIST or CIFAR-10
    if train_dataset[0][0].shape == torch.Size([1, 28, 28]):  # MNIST
        input_dim = 28 * 28  # 784
        num_classes = 10
    elif train_dataset[0][0].shape == torch.Size([3, 32, 32]):  # CIFAR-10
        input_dim = 3 * 32 * 32  # 3072
        num_classes = 10
    else:
        raise ValueError("Unsupported dataset shape")
    
    # Split train dataset into train and validation sets
    train_size = int(0.8 * len(train_dataset))
    val_size = len(train_dataset) - train_size
    train_dataset, val_dataset = random_split(train_dataset, [train_size, val_size])
    
    # Create DataLoader
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)
    
    # Model, criterion, and optimizer
    model = LinearRegressor(input_dim=input_dim)
    
    # Reduce the learning rate to avoid NaNs
    criterion = nn.MSELoss()
    optimizer = optim.SGD(model.parameters(), lr=0.001)  # Lower learning rate
    
    # Train the model
    best_model, final_model = train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=20)
    
    # Evaluate the model on the test set
    mse, r2, accuracy = evaluate_test(best_model, test_loader)

/var/folders/zx/c94b09b50v31fk6qp9h3h1cw0000gn/T/ipykernel_14525/3094469050.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  train_dataset = torch.load('datasets/cifar10_

Epoch [1/20], Loss: 15.7957, Val Loss: 10.2034
Epoch [2/20], Loss: 10.2152, Val Loss: 10.2418
Epoch [3/20], Loss: 9.1111, Val Loss: 12.7404
Epoch [4/20], Loss: 10.2022, Val Loss: 8.1624
Epoch [5/20], Loss: 9.0489, Val Loss: 8.5276
Epoch [6/20], Loss: 9.7235, Val Loss: 8.0019
Epoch [7/20], Loss: 9.2411, Val Loss: 8.1766
Epoch [8/20], Loss: 8.7263, Val Loss: 10.1782
Epoch [9/20], Loss: 8.8412, Val Loss: 8.9053
Epoch [10/20], Loss: 8.9599, Val Loss: 8.2524
Epoch [11/20], Loss: 9.7171, Val Loss: 7.8862
Epoch [12/20], Loss: 8.7759, Val Loss: 18.7115
Epoch [13/20], Loss: 9.6930, Val Loss: 10.3630
Epoch [14/20], Loss: 8.8407, Val Loss: 9.3781
Epoch [15/20], Loss: 8.8603, Val Loss: 7.8999
Epoch [16/20], Loss: 9.1345, Val Loss: 8.8140
Epoch [17/20], Loss: 8.9740, Val Loss: 8.0342
Epoch [18/20], Loss: 8.8862, Val Loss: 7.9230
Epoch [19/20], Loss: 8.8674, Val Loss: 10.0782
Epoch [20/20], Loss: 9.0388, Val Loss: 9.0923
Mean Squared Error: 9.2506
R² Score: -0.1213
Accuracy: 0.1144
